In [1]:
import os
import pickle
import json
from collections import Counter, defaultdict
from itertools import islice

In [3]:
word_file=r"F:/collage/NLP/Assignment_1/1/words.txt"

# genrelization

In [ ]:
# ---------- CONFIG ----------
TOKEN_FILE = r"F:/collage/NLP/Assignment_1/2/gu_meta_part_1.txt"  # file containing one token per line
# TOKEN_FILE = r"F:/collage/NLP/Assignment_1/1/words.txt"  # file containing one token per line
BATCH_SIZE = 7_00_000                                  # number of tokens to read per batch
MODEL_DIR = "ngram_model"
CHECKPOINT_FILE = os.path.join(MODEL_DIR, "checkpoint.pkl")
# ----------------------------

os.makedirs(MODEL_DIR, exist_ok=True)


# ---------- Data Loader ----------
def token_batches(file_path, batch_size, start_pos=0):
    """Yield batches of tokens with resuming support."""
    with open(file_path, "r", encoding="utf-8") as f:
        # Skip already processed lines if resuming
        for _ in range(start_pos):
            next(f, None)

        batch = []
        pos = start_pos
        for line in f:
            token = line.strip()
            if token:
                batch.append(token)
                pos += 1
                if len(batch) >= batch_size:
                    yield batch, pos
                    batch = []
        if batch:
            yield batch, pos


# ---------- Checkpoint Helpers ----------
def save_checkpoint(state):
    with open(CHECKPOINT_FILE, "wb") as f:
        pickle.dump(state, f)


def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "rb") as f:
            return pickle.load(f)
    return None


# ---------- Training ----------
def train_ngram_model(ngram_size: int):
    """
    Train an n-gram model of given size with checkpoint support.
    Saves:
        - final_ngram_counts.pkl
        - final_context_counts.pkl
        - final_ngram_probs.pkl
    """
    # Try to resume from checkpoint
    checkpoint = load_checkpoint()
    if checkpoint and checkpoint["ngram_size"] == ngram_size:
        print(f"[Resuming] {ngram_size}-gram training from checkpoint at position {checkpoint['pos']}")
        ngram_counts = checkpoint["ngram_counts"]
        context_counts = checkpoint["context_counts"]
        total_tokens = checkpoint["total_tokens"]
        start_pos = checkpoint["pos"]
        batch_no = checkpoint["batch_no"]
    else:
        print(f"[Starting Fresh] {ngram_size}-gram training")
        ngram_counts = defaultdict(int)
        context_counts = defaultdict(int)
        total_tokens = 0
        start_pos = 0
        batch_no = 0

    for batch, pos in token_batches(TOKEN_FILE, BATCH_SIZE, start_pos):
        for i in range(len(batch) - ngram_size + 1):
            ngram = tuple(batch[i:i + ngram_size])
            context = ngram[:-1] if ngram_size > 1 else ()
            ngram_counts[ngram] += 1
            context_counts[context] += 1

        batch_no += 1
        print(f"[Batch {batch_no}] Processed up to position {pos}")

        # For unigrams: maintain total token count
        if ngram_size == 1:
            total_tokens += len(batch)
            context_counts[()] = total_tokens

        # Save checkpoint after each batch
        save_checkpoint({
            "ngram_size": ngram_size,
            "ngram_counts": ngram_counts,
            "context_counts": context_counts,
            "total_tokens": total_tokens,
            "pos": pos,
            "batch_no": batch_no
        })
    print(f"[Checkpoint Saved] {ngram_size}-gram at batch {batch_no}, line {pos:,}")    

    # Save final counts
    with open(os.path.join(MODEL_DIR, f"final_{ngram_size}gram_counts.pkl"), "wb") as f:
        pickle.dump(dict(ngram_counts), f)
    with open(os.path.join(MODEL_DIR, f"final_{ngram_size}gram_context_counts.pkl"), "wb") as f:
        pickle.dump(dict(context_counts), f)

    print(f"[Training Complete] Final {ngram_size}-gram counts saved.")

    # ---- Compute probabilities ----
    ngram_probs = {}
    for ngram, count in ngram_counts.items():
        context = ngram[:-1] if ngram_size > 1 else ()
        context_count = context_counts[context]
        if context_count > 0:
            ngram_probs[ngram] = count / context_count

    # Save probabilities
    with open(os.path.join(MODEL_DIR, f"final_{ngram_size}gram_probs.pkl"), "wb") as f:
        pickle.dump(ngram_probs, f)

    print(f"[Probabilities Saved] {len(ngram_probs)} entries stored.")

    # Remove checkpoint after successful completion
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

    return ngram_counts, context_counts, ngram_probs


# ---------- Loader ----------
def load_ngram_model(ngram_size: int):
    """Load counts and probabilities for a trained n-gram model."""
    with open(os.path.join(MODEL_DIR, f"final_{ngram_size}gram_counts.pkl"), "rb") as f:
        ngram_counts = defaultdict(int, pickle.load(f))
    with open(os.path.join(MODEL_DIR, f"final_{ngram_size}gram_context_counts.pkl"), "rb") as f:
        context_counts = defaultdict(int, pickle.load(f))
    with open(os.path.join(MODEL_DIR, f"final_{ngram_size}gram_probs.pkl"), "rb") as f:
        ngram_probs = pickle.load(f)
    return ngram_counts, context_counts, ngram_probs





In [ ]:
# ---------- Example Run ----------
if __name__ == "__main__":
    for n in [1,2,3,4]:  # unigram, bigram, trigram, 4-gram
        print(f"\n--- Training {n}-gram model ---")
        counts, contexts, probs = train_ngram_model(n)
        print(f"Model size ({n}-gram): {len(counts)} ngrams, {len(probs)} probabilities")


--- Training 1-gram model ---
[Starting Fresh] 1-gram training
[Batch 1] Processed up to position 63785
[Checkpoint Saved] 1-gram at batch 1, line 63,785
[Training Complete] Final 1-gram counts saved.
[Probabilities Saved] 63785 entries stored.
Model size (1-gram): 63785 ngrams, 63785 probabilities


In [9]:
counts, contexts, probs = load_ngram_model(3)  # load bigram model
print(probs.get( ('.', 'આ', 'યુગો'), 1049))  # P("cat" | "the")


1049


# checking by loading

In [ ]:
file_path = r'F:/collage/NLP/Assignment_4/test_models/ngram_model/final_1gram_counts.pkl' 

try:
    # Open the file in binary read mode
    with open(file_path, 'rb') as file:
        # Load the pickled object from the file
        contex = pickle.load(file)
        
    
    print("Pickle file loaded successfully!")
    print("Loaded object:", contex)

except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found.")
except Exception as e:
    print(f"An error occurred while loading the pickle file: {e}")    

In [ ]:
# file_path = r'F:/collage/NLP/Assignment_4/test_models/ngram_model/final_2gram_context_counts.pkl' 

# try:
#     # Open the file in binary read mode
#     with open(file_path, 'rb') as file:
#         # Load the pickled object from the file
#         contex = pickle.load(file)
        
    
#     print("Pickle file loaded successfully!")
#     print("Loaded object:", contex)

# except FileNotFoundError:
#     print(f"Error: The file '{file_path}' was not found.")
# except Exception as e:
#     print(f"An error occurred while loading the pickle file: {e}")   

Pickle file loaded successfully!
Loaded object: {('દાન',): 3, ('આપો',): 4, ('ખાતું',): 3, ('બનાવો',): 3, ('પ્રવેશ',): 5, ('વિસર્જન',): 3, ('ગુજરાતીમાં',): 18, ('ટાઈપ',): 3, ('કરવા',): 3, ('માટે',): 25, ('ડાબી',): 3, ('બાજુના',): 3, ('હાંસિયામાં',): 3, ('કે',): 30, ('લેખની',): 3, ('ઉપર',): 4, ('ભાષાઓ',): 13, ('Languages',): 7, ('પર',): 38, ('તેની',): 10, ('બાજુમાં',): 3, ('રહેલા',): 5, ('ક્લિક',): 3, ('કરી',): 15, ('Input',): 3, ('માં',): 9, ('ગુજરાતી',): 246, ('હેઠળ',): 11, ('તમને',): 3, ('અનુકૂળ',): 3, ('કીબોર્ડ',): 3, ('પસંદ',): 3, ('કરો',): 8, ('.',): 936, ('વિગતો',): 2, ('છુપાવો',): 4, ('શરૂઆત',): 2, ('ઇતિહાસ',): 9, ('જૂની',): 16, ('ઈ',): 36, ('સ',): 50, ('૧૧૦૦',): 6, ('-',): 225, ('૧૫૦૦',): 12, ('મધ્યકાળની',): 10, ('૧૮૦૦',): 10, ('આધુનિક',): 48, ('અત્યારે',): 4, ('વસ્તીવિષયક',): 4, ('અને',): 314, ('વિતરણ',): 8, ('સંદર્ભ',): 6, ('બાહ્ય',): 7, ('કડીઓ',): 5, ('ભાષા',): 70, ('લેખ',): 9, ('ચર્ચા',): 5, ('વાંચો',): 3, ('ફેરફાર',): 11, ('જુઓ',): 6, ('સાધનો',): 3, ('દેખાવ',): 4, ('લખાણ',)

In [ ]:
# file_path = r'F:/collage/NLP/Assignment_4/test_models/ngram_model/final_2gram_probs.pkl' 

# try:
#     # Open the file in binary read mode
#     with open(file_path, 'rb') as file:
#         # Load the pickled object from the file
#         contex = pickle.load(file)
        
    
#     print("Pickle file loaded successfully!")
#     print("Loaded object:", contex)

# except FileNotFoundError:
#     print(f"Error: The file '{file_path}' was not found.")
# except Exception as e:
#     print(f"An error occurred while loading the pickle file: {e}")   

Pickle file loaded successfully!
Loaded object: {('દાન', 'આપો'): 1.0, ('આપો', 'ખાતું'): 0.75, ('ખાતું', 'બનાવો'): 1.0, ('બનાવો', 'પ્રવેશ'): 1.0, ('પ્રવેશ', 'વિસર્જન'): 0.6, ('વિસર્જન', 'ગુજરાતીમાં'): 1.0, ('ગુજરાતીમાં', 'ટાઈપ'): 0.16666666666666666, ('ટાઈપ', 'કરવા'): 1.0, ('કરવા', 'માટે'): 1.0, ('માટે', 'ડાબી'): 0.12, ('ડાબી', 'બાજુના'): 1.0, ('બાજુના', 'હાંસિયામાં'): 1.0, ('હાંસિયામાં', 'કે'): 1.0, ('કે', 'લેખની'): 0.1, ('લેખની', 'ઉપર'): 1.0, ('ઉપર', 'ભાષાઓ'): 0.75, ('ભાષાઓ', 'કે'): 0.23076923076923078, ('કે', 'Languages'): 0.1, ('Languages', 'પર'): 0.42857142857142855, ('પર', 'કે'): 0.07894736842105263, ('કે', 'તેની'): 0.1, ('તેની', 'બાજુમાં'): 0.3, ('બાજુમાં', 'રહેલા'): 1.0, ('રહેલા', 'પર'): 0.6, ('પર', 'ક્લિક'): 0.07894736842105263, ('ક્લિક', 'કરી'): 1.0, ('કરી', 'Input'): 0.2, ('Input', 'માં'): 1.0, ('માં', 'ગુજરાતી'): 0.5555555555555556, ('ગુજરાતી', 'હેઠળ'): 0.012195121951219513, ('હેઠળ', 'તમને'): 0.2727272727272727, ('તમને', 'અનુકૂળ'): 1.0, ('અનુકૂળ', 'કીબોર્ડ'): 1.0, ('કીબોર્ડ'

In [ ]:
# file_path = r'F:/collage/NLP/Assignment_4/test_models/ngram_model/final_4gram_probs.pkl' 

# try:
#     # Open the file in binary read mode
#     with open(file_path, 'rb') as file:
#         # Load the pickled object from the file
#         ngram = pickle.load(file)
        
    
#     print("Pickle file loaded successfully!")
#     print("Loaded object:", ngram)

# except FileNotFoundError:
#     print(f"Error: The file '{file_path}' was not found.")
# except Exception as e:
#     print(f"An error occurred while loading the pickle file: {e}")    

# print(ngram)   

Pickle file loaded successfully!
Loaded object: {('દાન', 'આપો', 'ખાતું', 'બનાવો'): 1.0, ('આપો', 'ખાતું', 'બનાવો', 'પ્રવેશ'): 1.0, ('ખાતું', 'બનાવો', 'પ્રવેશ', 'વિસર્જન'): 1.0, ('બનાવો', 'પ્રવેશ', 'વિસર્જન', 'ગુજરાતીમાં'): 1.0, ('પ્રવેશ', 'વિસર્જન', 'ગુજરાતીમાં', 'ટાઈપ'): 1.0, ('વિસર્જન', 'ગુજરાતીમાં', 'ટાઈપ', 'કરવા'): 1.0, ('ગુજરાતીમાં', 'ટાઈપ', 'કરવા', 'માટે'): 1.0, ('ટાઈપ', 'કરવા', 'માટે', 'ડાબી'): 1.0, ('કરવા', 'માટે', 'ડાબી', 'બાજુના'): 1.0, ('માટે', 'ડાબી', 'બાજુના', 'હાંસિયામાં'): 1.0, ('ડાબી', 'બાજુના', 'હાંસિયામાં', 'કે'): 1.0, ('બાજુના', 'હાંસિયામાં', 'કે', 'લેખની'): 1.0, ('હાંસિયામાં', 'કે', 'લેખની', 'ઉપર'): 1.0, ('કે', 'લેખની', 'ઉપર', 'ભાષાઓ'): 1.0, ('લેખની', 'ઉપર', 'ભાષાઓ', 'કે'): 1.0, ('ઉપર', 'ભાષાઓ', 'કે', 'Languages'): 1.0, ('ભાષાઓ', 'કે', 'Languages', 'પર'): 1.0, ('કે', 'Languages', 'પર', 'કે'): 1.0, ('Languages', 'પર', 'કે', 'તેની'): 1.0, ('પર', 'કે', 'તેની', 'બાજુમાં'): 1.0, ('કે', 'તેની', 'બાજુમાં', 'રહેલા'): 1.0, ('તેની', 'બાજુમાં', 'રહેલા', 'પર'): 1.0, ('બાજુમાં', 

# count to prob

In [ ]:
# def ngram_probability(ngram, ngram_counts, context_counts):
#     """
#     Compute probability of the last word in the ngram given its context.
#     P(w_n | w_1,...,w_{n-1}) = Count(ngram) / Count(context)
#     """
#     context = ngram[:-1]
#     ngram_count = ngram_counts.get(ngram, 0)
#     context_count = context_counts.get(context, 0)

#     if context_count == 0:
#         return 0.0
#     return ngram_count / context_count


In [ ]:
# def sentence_probability(sentence_tokens, ngram_counts, context_counts, n):
#     """
#     Compute the probability of a full sentence using an n-gram model.
#     """
#     # Add start/end tokens for proper padding
#     padded_tokens = ["<s>"]*(n-1) + sentence_tokens + ["</s>"]
#     prob = 1.0

#     for i in range(len(padded_tokens) - n + 1):
#         ngram = tuple(padded_tokens[i:i+n])
#         p = ngram_probability(ngram, ngram_counts, context_counts)
#         if p == 0:   # unseen n-gram
#             return 0.0
#         prob *= p

#     return prob


In [ ]:
# ngram_probability( ('ટાઈપ', 'કરવા'),ngram,contex)

In [ ]:
# sentence_probability( ['ટાઈપ', 'કરવા'],ngram,contex,n=2)